# シン・ヒョンジ｜分類テスト・バックテスト：急騰予測の最終評価

`baseline_analysis.ipynb`で決めた設定（比率特徴量4つ、正則化、しきい値0.8、ルール＋モデル）を変更せずに使用します。テスト期間（2025.11～2026.10）は、このNotebookで**一度だけ**使用します。

## 0. 準備

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip -q install japanize-matplotlib
import japanize_matplotlib   # グラフの日本語文字化けを防ぐ

from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score, precision_score, average_precision_score, precision_recall_curve

BASE = "/content/drive/MyDrive/kmove 2차 프로젝트/산출물 관리"     # 結果の保存先
df = pd.read_csv(f"{BASE}/D02_전처리&정제 데이터/merged_30min.csv", parse_dates=["date"])
df = df.sort_values(["date", "slot"]).reset_index(drop=True)
print(df.shape)   # (45312, 73)

In [ ]:
# ベースラインで決めたXと設定をそのまま使用
drop = ["date", "slot", "slot_start", "split", "y_price", "y_spike_rel",
        "has_occto", "lng_yen_per_t_m1", "regime_2026"]
df["ratio_lag1d"]  = df["price_tokyo_lag1d"]  / df["price_tokyo_mean30d"]
df["ratio_lag7d"]  = df["price_tokyo_lag7d"]  / df["price_tokyo_mean30d"]
df["ratio_mean7d"] = df["price_tokyo_mean7d"] / df["price_tokyo_mean30d"]
df["spike_lag1d"]  = (df["ratio_lag1d"] >= 1.5).astype(int)
X_cols = [c for c in df.columns if c not in drop]
colsA  = [c for c in X_cols if not c.startswith("occto_")]
params = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=100,
              reg_lambda=1.0, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              class_weight="balanced", importance_type="gain", random_state=42, verbose=-1)
print("特徴量Xの数：モデルA", len(colsA), "/ モデルB", len(X_cols))

## 8. 最終学習 → テスト予測 → 保存（未実行）

パラメータとしきい値（0.8）は検証段階で決定した設定のまま使用します。**学習＋検証期間を使って再学習**した後、テスト（2025.11～2026.10）に一度だけ適用します。

- モデルA：2024.4～2025.10／モデルB：2025.4～2025.10
- 保存形式（チーム共通）：`date, slot, split, y_price, y_price_pred, y_spike_rel, spike_prob, spike_pred`＋参考列。`y_price_pred`は回帰担当が入力します。

In [ ]:
trA_f = df[df["split"] != "test"]
trB_f = df[(df["has_occto"] == 1) & (df["split"] != "test")]
te    = df[df["split"] == "test"].copy()

mA_f = LGBMClassifier(**params).fit(trA_f[colsA],  trA_f["y_spike_rel"])
mB_f = LGBMClassifier(**params).fit(trB_f[X_cols], trB_f["y_spike_rel"])
te["prob_A"] = mA_f.predict_proba(te[colsA])[:, 1]
te["prob_B"] = mB_f.predict_proba(te[X_cols])[:, 1]

te["alarm_rule"] = (te["ratio_lag1d"] >= 1.5).astype(int)
te["alarm_A"] = ((te["alarm_rule"] == 1) | (te["prob_A"] >= 0.8)).astype(int)
te["alarm_B"] = ((te["alarm_rule"] == 1) | (te["prob_B"] >= 0.8)).astype(int)

y = te["y_spike_rel"]
print(f"[テスト] 急騰 {y.sum()}件 / ランダム予測ベースライン {y.mean():.3f}")
print(f"PR-AUC  モデルA {average_precision_score(y, te['prob_A']):.3f}  /  モデルB {average_precision_score(y, te['prob_B']):.3f}")
for k, name in [("alarm_rule", "ルールのみ"), ("alarm_A", "ルール＋A≥0.8"), ("alarm_B", "ルール＋B≥0.8")]:
    print(f"{name}: 再現率 {recall_score(y, te[k]):.3f}, 適合率 {precision_score(y, te[k]):.3f}, アラート {te[k].sum()}")

out = te[["date", "slot", "split", "y_price", "y_spike_rel"]].copy()
out["y_price_pred"] = np.nan
out["spike_prob"] = te["prob_B"]
out["spike_pred"] = te["alarm_B"]
out[["prob_A", "alarm_rule", "alarm_A"]] = te[["prob_A", "alarm_rule", "alarm_A"]]
out = out[["date", "slot", "split", "y_price", "y_price_pred", "y_spike_rel",
           "spike_prob", "spike_pred", "prob_A", "alarm_rule", "alarm_A"]]
out.to_csv(f"{BASE}/test_predictions.csv", index=False)
print("保存完了：", out.shape)

## 9. バックテスト：年間の電力調達費削減額（未実行）

- 仮想企業が30分ごとに東京エリア需要の0.1%を購入すると想定します。実際の需要には、翌日の行にある`demand_mw_lag1d`を使用します（**結果算出のみに使用し、予測特徴量としては使わない**）。
- アラートが出た枠では代替単価で、それ以外では市場価格で購入します。**削減額＝すべて市場で購入した場合の費用－戦略適用後の費用**
- 代替単価：**基本案＝直近30日間の平均価格×倍率**、比較案＝学習期間の平均価格×倍率（2026年の価格上昇を踏まえ、固定単価には限界があります）
- 倍率1.1／1.2／1.5で感度分析します。

In [ ]:
df["demand_actual_mw"] = df["demand_mw_lag1d"].shift(-48)
te["demand_actual_mw"] = df.loc[te.index, "demand_actual_mw"].fillna(te["demand_mw_lag1d"])
kwh  = te["demand_actual_mw"] * 0.5            # MW × 0.1% × 1000(kW) × 0.5時間
spot = (kwh * te["y_price"]).sum()
train_mean = df.loc[df["split"] == "train", "y_price"].mean()

strategies = {"常に代替調達": pd.Series(1, index=te.index), "ルールのみ": te["alarm_rule"],
              "ルール＋A": te["alarm_A"], "ルール＋B": te["alarm_B"], "完全な予測": y}
rows = []
for alt_name, base_price in [("変動：30日間平均を基準", te["price_tokyo_mean30d"]), ("固定：学習期間平均を基準", train_mean)]:
    for mult in [1.1, 1.2, 1.5]:
        alt = base_price * mult
        row = {"代替単価": alt_name, "倍率": mult}
        for k, a in strategies.items():
            cost = (kwh * np.where(a == 1, alt, te["y_price"])).sum()
            row[k] = round((spot - cost) / 1e6, 1)       # 百万円
        rows.append(row)
bt = pd.DataFrame(rows)
print(f"すべて市場で購入した場合の年間費用： {spot/1e8:.2f}億円  (削減額の単位：百万円)")
display(bt)
bt.to_csv(f"{BASE}/backtest_results.csv", index=False)

In [ ]:
r = bt[(bt["代替単価"].str.startswith("変動")) & (bt["倍率"] == 1.2)].iloc[0]
names = ["ルールのみ", "ルール＋A", "ルール＋B", "完全な予測"]
plt.figure(figsize=(7, 4))
bars = plt.bar(names, [r[n] for n in names], color=["#888780", "#378ADD", "#185FA5", "#B4B2A9"])
plt.bar_label(bars, fmt="%.0f")
plt.title("年間の調達費削減額（百万円）・代替単価＝30日間平均×1.2")
plt.tight_layout()
plt.savefig(f"{BASE}/backtest_savings.png", dpi=150)
plt.show()